# Myllia: Direction Notebook (Bilinear Conditional Factorization)

This notebook implements a medium-large architecture shift:

- Learn a low-rank interaction between perturbed gene embeddings and output gene embeddings
- Predict the full delta vector as a bilinear form with rank `R`
- Train with a metric-aligned weighted L1 proxy and evaluate with the official `myllia_score`

Core model:
\[
\hat D_{i,j} = \langle W_p z_{g_i},\; W_o u_j \rangle + b_j + b_i
\]
where:
- `z_{g_i}` is an embedding for the perturbed gene `g_i`
- `u_j` is an embedding for output gene `j`
- `R` is a small rank (16 to 64)

This uses `training_cells.h5ad` to build output gene embeddings.


In [444]:
import numpy as np
import pandas as pd
from pathlib import Path

import anndata as ad
import scanpy as sc
from scipy import sparse

from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

import torch
import torch.nn as nn

from myllia_metric import myllia_score

SEED = 6
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)

EMB_DIM_PERT = 128   # pert gene embedding dim (from SVD)
EMB_DIM_OUT = 128   # output gene embedding dim (from SVD)
RANK_R = 32    # low-rank interaction size

DROPOUT = 0.10
LR = 2e-3 * 0.6 * 1.2
WD = 1e-4 * 1.5
EPOCHS = 400
BATCH_GENES = 16 # minibatch over perturbed genes
EVAL_EVERY = 25
PATIENCE = 12 # early stopping patience in eval steps

# Gate parameters should match metric structure
GATE_A = 0.0
GATE_B = 0.2
EPS = 1e-12

ROOT = Path(".")

# --- added: CV alpha sweep + refit ---
MODEL_SEEDS = [6, 7, 8]
ALPHA_GRID = [0.7, 0.75, 0.778, 0.82, 0.86]
GRAD_CLIP = 1.0
H5AD_PATH = ROOT / "data" / "training_cells.h5ad"  # used ONLY for perts not in gene_columns


In [445]:
def score_delta(dt, dp):
    dt = dt.astype(np.float32, copy=False)
    dp = dp.astype(np.float32, copy=False)
    r = myllia_score(dt, dp)
    return {
        "score": float(r.score),
        "wcos": float(r.wcos),
        "mean_term": float(r.mean_term),
        "pred_wmae": float(r.pred_wmae),
    }


In [446]:
means_path = ROOT / "data" / "training_data_means.csv"
valmap_path = ROOT / "data" / "pert_ids_val.csv"
sample_sub_path = ROOT / "data" / "sample_submission.csv"

df_means = pd.read_csv(means_path)
df_valmap = pd.read_csv(valmap_path)
df_sub = pd.read_csv(sample_sub_path)

gene_columns = [c for c in df_means.columns if c != "pert_symbol"]

baseline_mask = df_means["pert_symbol"].astype(str) == "non-targeting"
x_base = df_means.loc[baseline_mask, gene_columns].iloc[0].to_numpy(np.float32)

df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
train_genes = df_train["pert_symbol"].astype(str).to_numpy()

X_train_means = df_train[gene_columns].to_numpy(np.float32)
D_train = X_train_means - x_base[None, :] # (80, 5127) delta vs non-targeting

delta_baseline = D_train.mean(axis=0).astype(np.float32)

# pert_id -> gene symbol for leaderboard (first 60)
val_map = dict(zip(df_valmap["pert_id"].astype(str), df_valmap["pert"].astype(str)))

print("Train perts:", len(train_genes), "G:", len(gene_columns))
print("Sample submission rows:", len(df_sub))
print("Val mapping entries:", len(val_map))


Train perts: 80 G: 5127
Sample submission rows: 120
Val mapping entries: 60


In [447]:
EXP_NAME = "baseline_fixed_missing"

# Always fill missing perts from h5ad (DO NOT TURN OFF)
FILL_MISSING_PERTS = True
H5AD_TOPK = 256

# Experiment A: output gene embeddings from h5ad control cells
USE_UOUT_CTRL=True
UOUT_BLEND_ALPHA = 1   # 0 -> pure SVD U_out, 1 -> pure ctrl U_out
UOUT_CTRL_MODE="replace"  # "replace", "concat", or "blend"

# Experiment B: pert embeddings from h5ad control corr for ALL perts (blend with SVD)
USE_ZCTRL_ALL=True
ZCTRL_BETA = 0.50
ZCTRL_TOPK = 512

# Experiment C: GenePT
USE_GENEPT = True
GENEPT_FUSION = "concat"   # "blend" or "concat"
GENEPT_GAMMA = 0.10
GENEPT_WHERE = "u"  # "z", "u", or "both"
GENEPT_DIR = ROOT / "external" / "genept"
GENEPT_WHICH = "gene_protein"  # "gene" or "gene_protein"

# If concat makes training unstable, scale GenePT down
GENEPT_DIM_Z = 64
GENEPT_DIM_U = 64
GENEPT_SCALE_Z = 0.25
GENEPT_SCALE_U = 0.10
GENEPT_MODE = "svd"  # "svd" or "slice"

import pickle
import scipy.sparse as sp
from sklearn.preprocessing import StandardScaler

_H5AD_CACHE = None

def load_genept_embeddings(genept_dir="external/genept", which="gene_protein"):
    genept_dir = Path(genept_dir)

    if which == "gene":
        fname = "GenePT_gene_embedding_ada_text.pickle"
    elif which == "gene_protein":
        fname = "GenePT_gene_protein_embedding_model_3_text.pickle"
    else:
        raise ValueError(f"unknown which={which}")

    with open(genept_dir / fname, "rb") as f:
        d = pickle.load(f)

    # Normalize keys and values
    out = {}
    for k, v in d.items():
        kk = str(k).upper()
        out[kk] = np.asarray(v, dtype=np.float32)
    return out

def l2norm_rows(X, eps=1e-12):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)

def reduce_gene_embeddings(g2v, genesU, out_dim, mode="svd", seed=6, l2norm=True):
    """
    g2v: dict {SYMBOL_UPPER: vec}
    genesU: iterable of gene identifiers (already uppercase symbols ideally)
    out_dim: desired reduced dimension
    mode: "svd" or "slice"
    returns: (dict {gene: reduced_vec}, fallback_vec)
    """
    avail = [g for g in genesU if g in g2v]
    if len(avail) < 10:
        print("[genept] too few genes available for reduction:", len(avail))
        return None, None

    X = np.stack([np.asarray(g2v[g], dtype=np.float32) for g in avail], axis=0)

    # Slice mode: cheap, no fit, but usually worse than SVD
    if mode == "slice":
        d = min(int(out_dim), int(X.shape[1]))
        Z = X[:, :d].astype(np.float32)
        if d < out_dim:
            Z = np.concatenate([Z, np.zeros((Z.shape[0], out_dim - d), dtype=np.float32)], axis=1)
        if l2norm:
            Z = l2norm_rows(Z)
        fb = Z.mean(axis=0).astype(np.float32)
        return {avail[i]: Z[i] for i in range(len(avail))}, fb

    # SVD mode: standardize then reduce
    Xs = StandardScaler(with_mean=True, with_std=True).fit_transform(X).astype(np.float32)
    n, d = Xs.shape

    # Clamp components: k must be <= min(n-1, d)
    k = min(int(out_dim), int(d), int(n - 1))
    if k < 1:
        print("[genept] cannot reduce: n, d =", n, d)
        return None, None

    svd = TruncatedSVD(n_components=k, random_state=seed)
    Zk = svd.fit_transform(Xs).astype(np.float32)

    # Pad back up if you want exactly out_dim
    if k < out_dim:
        Z = np.concatenate([Zk, np.zeros((n, out_dim - k), dtype=np.float32)], axis=1)
    else:
        Z = Zk

    if l2norm:
        Z = l2norm_rows(Z)

    fb = Z.mean(axis=0).astype(np.float32)
    return {avail[i]: Z[i] for i in range(len(avail))}, fb

def _pick_pert_col(adata):
    for c in ["sgrna_symbol", "pert_symbol", "pert", "perturbation", "gene", "target_gene"]:
        if c in adata.obs.columns:
            return c
    raise ValueError("Could not find perturbation column in h5ad obs.")

def load_h5ad_ctrl_cache(h5ad_path: Path, gene_columns):
    """
    Cache:
      - Xn (CPM10K + log2(1+x)) sparse
      - ctrl mask
      - var map (GENE->idx)
      - out_idx mapping for the 5127 output genes
      - Xout_z (control cells, standardized per gene)
    """
    global _H5AD_CACHE
    if _H5AD_CACHE is not None:
        return _H5AD_CACHE

    adata = ad.read_h5ad(str(h5ad_path))
    pert_col = _pick_pert_col(adata)

    Xc = adata.X
    if not sp.issparse(Xc):
        Xc = sp.csr_matrix(Xc)
    else:
        Xc = Xc.tocsr()

    cell_sum = np.asarray(Xc.sum(axis=1)).ravel().astype(np.float64)
    scale = (10000.0 / np.clip(cell_sum, 1e-12, None)).astype(np.float64)
    Xn = Xc.multiply(scale[:, None]).tocsr()
    Xn.data = np.log1p(Xn.data) / np.log(2.0)

    obs_pert = adata.obs[pert_col].astype(str).to_numpy()
    obs_pertU = np.char.upper(obs_pert.astype("U"))

    ctrl_mask = (obs_pertU == "NON-TARGETING")
    if int(ctrl_mask.sum()) == 0:
        raise ValueError("No non-targeting control cells found in h5ad.")

    var_names = adata.var_names.astype(str).to_numpy()
    varU = np.char.upper(var_names.astype("U"))
    var = {varU[i]: i for i in range(len(varU))}

    out_idx = np.array([var[str(g).upper()] for g in gene_columns], dtype=np.int64)

    Xn_ctrl = Xn[ctrl_mask]
    Xout = Xn_ctrl[:, out_idx]
    if sp.issparse(Xout):
        Xout = Xout.toarray()
    Xout = Xout.astype(np.float32)

    mu = Xout.mean(axis=0, keepdims=True)
    sd = Xout.std(axis=0, keepdims=True) + 1e-6
    Xout_z = ((Xout - mu) / sd).astype(np.float32)

    _H5AD_CACHE = dict(
        Xn=Xn,
        Xn_ctrl=Xn_ctrl,
        ctrl_mask=ctrl_mask,
        var=var,
        out_idx=out_idx,
        Xout_z=Xout_z,
        obs_pertU=obs_pertU,
    )
    print("[h5ad] cache built:", "n_cells=", Xn.shape[0], "n_genes=", Xn.shape[1], "n_ctrl=", int(ctrl_mask.sum()))
    return _H5AD_CACHE

def build_u_out_from_ctrl(cache, d_out, seed=SEED):
    svd_ctrl = TruncatedSVD(n_components=d_out, random_state=seed)
    svd_ctrl.fit(cache["Xout_z"])
    return svd_ctrl.components_.T.astype(np.float32)  # (G, d_out)

def build_z_from_ctrl_corr(cache, genesU, P_out, topk=256):
    Xout_z = cache["Xout_z"]
    Xn_ctrl = cache["Xn_ctrl"]
    var = cache["var"]

    out = {}
    for gU in genesU:
        j = var.get(gU, None)
        if j is None:
            continue

        xg = Xn_ctrl[:, j]
        if sp.issparse(xg):
            xg = xg.toarray()
        xg = np.asarray(xg).ravel().astype(np.float32)
        xg = (xg - xg.mean()) / (xg.std() + 1e-6)

        corr = (xg[:, None] * Xout_z).mean(axis=0)
        idx = np.argsort(-np.abs(corr))[:topk]
        w = corr[idx].astype(np.float32)

        z = (w[:, None] * P_out[idx]).sum(axis=0)
        z = z / (np.linalg.norm(z) + 1e-12)
        out[gU] = z.astype(np.float32)

    return out

def build_genept_reduced_tables(genept_dict, genes_for_z, genes_for_u, dim_z, dim_u, seed=6,
                               reducer_mode="svd", l2norm=True):
    z_table, z_fb = reduce_gene_embeddings(genept_dict, genes_for_z, dim_z, mode=reducer_mode, seed=seed, l2norm=l2norm)
    u_table, u_fb = reduce_gene_embeddings(genept_dict, genes_for_u, dim_u, mode=reducer_mode, seed=seed, l2norm=l2norm)
    return z_table, z_fb, u_table, u_fb

def concat_genept(base_vec, genept_vec, scale=0.25, l2_after=False):
    if genept_vec is None:
        out = base_vec
    else:
        out = np.concatenate([base_vec, scale * genept_vec], axis=-1)
    if l2_after:
        out = out / (np.linalg.norm(out) + 1e-12)
    return out.astype(np.float32)

Build gene embeddings from D_train (SVD on signed-log transformed deltas). Build embeddings for output genes (gene_columns) via SVD on (80, 5127). Pert embeddings are looked up by gene symbol in gene_columns, otherwise fallback to the mean embedding.

In [448]:
val_targets = df_valmap["pert"].astype(str).tolist()

genes_needed = sorted(set([str(g).upper() for g in train_genes.tolist()] +
                          [str(g).upper() for g in val_targets]))

geneU = pd.Index([str(g).upper() for g in gene_columns])

# signed log transform (handles negative deltas)
X = D_train.astype(np.float32, copy=True)
X = np.sign(X) * np.log2(1.0 + np.abs(X))

svd = TruncatedSVD(n_components=max(EMB_DIM_PERT, EMB_DIM_OUT), random_state=SEED)
svd.fit(X)

gene_emb_all = svd.components_.T.astype(np.float32)  # (G, k)
k_svd = gene_emb_all.shape[1]
d_pert = min(int(EMB_DIM_PERT), int(k_svd))
d_out  = min(int(EMB_DIM_OUT),  int(k_svd))

print("SVD k:", k_svd, "gene_emb_all:", gene_emb_all.shape)
print("Effective dims:", "d_pert=", d_pert, "d_out=", d_out)

# base dictionaries (only for genes in gene_columns)
gene2emb_pert_svd = {gene_columns[i].upper(): gene_emb_all[i, :d_pert].copy()
                     for i in range(len(gene_columns))}
gene2emb_out_svd  = {gene_columns[i].upper(): gene_emb_all[i, :d_out ].copy()
                     for i in range(len(gene_columns))}

emb_fallback_pert = gene_emb_all[:, :d_pert].mean(axis=0).astype(np.float32)
emb_fallback_out  = gene_emb_all[:, :d_out ].mean(axis=0).astype(np.float32)

# Missing perts lists (these are the ones that were killing fairness)
missing_train = [g for g in train_genes.tolist() if str(g).upper() not in geneU]
missing_val   = [g for g in val_targets           if str(g).upper() not in geneU]
missing_allU  = sorted(set([str(g).upper() for g in (missing_train + missing_val)]))

if missing_train:
    print(f"train perts not in gene_columns: {len(missing_train)}. Example: {missing_train[:12]}")
if missing_val:
    print(f"val perts not in gene_columns: {len(missing_val)}. Example: {missing_val[:12]}")

# ---------------------------------
# h5ad: build embeddings
# ---------------------------------
missing_pert_h5ad = {}
zctrl_all = {}

U_ctrl = None

if (FILL_MISSING_PERTS and len(missing_allU) > 0) or USE_ZCTRL_ALL or USE_UOUT_CTRL:
    cache = load_h5ad_ctrl_cache(H5AD_PATH, gene_columns)
    P_out = gene_emb_all[:, :d_pert].astype(np.float32)  # (5127, d_pert) basis

    # Fill missing perts (critical)
    if FILL_MISSING_PERTS and len(missing_allU) > 0:
        tmp = build_z_from_ctrl_corr(cache, missing_allU, P_out, topk=H5AD_TOPK)
        missing_pert_h5ad.update(tmp)
        print("[h5ad] embedded missing perts:", len(missing_pert_h5ad), "of", len(missing_allU))

    # Optional: build h5ad corr for ALL perts (for blending experiment)
    if USE_ZCTRL_ALL:
        tmp = build_z_from_ctrl_corr(cache, genes_needed, P_out, topk=ZCTRL_TOPK)
        zctrl_all.update(tmp)
        print("[h5ad] embedded ALL perts for blending:", len(zctrl_all), "of", len(genes_needed))

    # Optional: U_out from control cells (now actually used)
    if USE_UOUT_CTRL:
        U_ctrl = build_u_out_from_ctrl(cache, d_out=d_out, seed=SEED)  # (5127, d_out)
        print("[h5ad] U_ctrl:", U_ctrl.shape)

# ---------------------------------
# GenePT: load + reduce (FIXED: dims + BOTH support)
# ---------------------------------
genept_red_z = None
genept_fb_z  = None
genept_red_u = None
genept_fb_u  = None

if USE_GENEPT:
    g2v = load_genept_embeddings(GENEPT_DIR, which=GENEPT_WHICH)

    # Decide reduction dims based on fusion mode:
    # - blend: must match existing dims
    # - concat: user-defined dims
    if GENEPT_FUSION == "blend":
        dim_z = int(d_pert)
        dim_u = int(d_out)
    elif GENEPT_FUSION == "concat":
        dim_z = int(GENEPT_DIM_Z)
        dim_u = int(GENEPT_DIM_U)
    else:
        raise ValueError("GENEPT_FUSION must be 'blend' or 'concat'")

    if (GENEPT_WHERE in ["z", "both"]):
        genept_red_z, genept_fb_z = reduce_gene_embeddings(
            g2v, genes_needed, out_dim=dim_z, mode=GENEPT_MODE, seed=SEED
        )

    if (GENEPT_WHERE in ["u", "both"]):
        genept_red_u, genept_fb_u = reduce_gene_embeddings(
            g2v, [str(g).upper() for g in gene_columns], out_dim=dim_u, mode=GENEPT_MODE, seed=SEED
        )

    print("[genept] reduced:",
          "z_avail=", (0 if genept_red_z is None else len(genept_red_z)),
          "u_avail=", (0 if genept_red_u is None else len(genept_red_u)),
          "dim_z=", (None if genept_fb_z is None else genept_fb_z.shape[0]),
          "dim_u=", (None if genept_fb_u is None else genept_fb_u.shape[0]))

# ---------------------------------
# Final emb_pert (FIXED: GenePT supports 'both')
# ---------------------------------
def emb_pert(g: str) -> np.ndarray:
    gU = str(g).upper()

    # Base: SVD if gene in outputs, else h5ad-missing if available, else fallback mean
    if gU in gene2emb_pert_svd:
        z_base = gene2emb_pert_svd[gU]
    elif gU in missing_pert_h5ad:
        z_base = missing_pert_h5ad[gU]
    else:
        z_base = emb_fallback_pert

    # Optional: blend with h5ad corr for all perts (if available)
    if USE_ZCTRL_ALL and (gU in zctrl_all):
        z = (1.0 - float(ZCTRL_BETA)) * z_base + float(ZCTRL_BETA) * zctrl_all[gU]
    else:
        z = z_base

    # GenePT for Z (supports "z" OR "both")
    if USE_GENEPT and (GENEPT_WHERE in ["z", "both"]) and (genept_red_z is not None):
        gp = genept_red_z.get(gU, genept_fb_z)
        if gp is not None:
            gp = gp.astype(np.float32)

            if GENEPT_FUSION == "concat":
                z = np.concatenate([z, float(GENEPT_SCALE_Z) * gp], axis=0)

            elif GENEPT_FUSION == "blend":
                gg = float(GENEPT_GAMMA)
                z = (1.0 - gg) * z + gg * gp
                z = z / (np.linalg.norm(z) + 1e-12)

            else:
                raise ValueError("GENEPT_FUSION must be 'blend' or 'concat'")

    return z.astype(np.float32)

# ---------------------------------
# Build U_out (base + ctrl) first
# ---------------------------------
U_out_base = np.vstack([gene2emb_out_svd.get(str(g).upper(), emb_fallback_out) for g in gene_columns]).astype(np.float32)

if USE_UOUT_CTRL and (U_ctrl is not None):
    if UOUT_CTRL_MODE == "replace":
        U_out = U_ctrl.astype(np.float32)
        print("[exp] U_out = ctrl (replace)")

    elif UOUT_CTRL_MODE == "concat":
        U_out = np.concatenate([U_out_base, U_ctrl.astype(np.float32)], axis=1)
        print("[exp] U_out = concat(base, ctrl)")

    elif UOUT_CTRL_MODE == "blend":
        a = float(UOUT_BLEND_ALPHA)
        U_out = (1.0 - a) * U_out_base + a * U_ctrl.astype(np.float32)
        U_out = U_out / (np.linalg.norm(U_out, axis=1, keepdims=True) + 1e-12)
        print(f"[exp] U_out = blend alpha={a:.3f} (SVD+ctrl)")

    else:
        raise ValueError("UOUT_CTRL_MODE must be 'replace', 'concat', or 'blend'")
else:
    U_out = U_out_base
    print("[exp] U_out = base SVD")

# ---------------------------------
# GenePT for U_out (FIXED: blend OR concat; supports "u" OR "both")
# ---------------------------------
if USE_GENEPT and (GENEPT_WHERE in ["u", "both"]) and (genept_red_u is not None):
    U_gp = np.vstack([genept_red_u.get(str(g).upper(), genept_fb_u) for g in gene_columns]).astype(np.float32)

    if GENEPT_FUSION == "concat":
        U_out = np.concatenate([U_out, float(GENEPT_SCALE_U) * U_gp], axis=1).astype(np.float32)
        print("[exp] U_out = concat(+GenePT) ->", U_out.shape)

    elif GENEPT_FUSION == "blend":
        gg = float(GENEPT_GAMMA)
        U_out = (1.0 - gg) * U_out + gg * U_gp
        U_out = U_out / (np.linalg.norm(U_out, axis=1, keepdims=True) + 1e-12)
        print(f"[exp] U_out = blend(+GenePT) gamma={gg:.3f} ->", U_out.shape)

    else:
        raise ValueError("GENEPT_FUSION must be 'blend' or 'concat'")

# Pert embeddings for the 80 training perts
Z_train = np.vstack([emb_pert(g) for g in train_genes]).astype(np.float32)

print("EXP_NAME:", EXP_NAME)
print("U_out:", U_out.shape, "Z_train:", Z_train.shape)

SVD k: 80 gene_emb_all: (5127, 80)
Effective dims: d_pert= 80 d_out= 80
train perts not in gene_columns: 8. Example: ['BRD4', 'CHD4', 'DNAJA3', 'INO80', 'KAT8', 'KDM4A', 'PMEL', 'SETD1A']
val perts not in gene_columns: 8. Example: ['SMARCB1', 'PSMA1', 'CUL1', 'FLT4', 'FOXH1', 'HK2', 'TRAM2', 'DPH2']
[h5ad] cache built: n_cells= 17882 n_genes= 19226 n_ctrl= 1026
[h5ad] embedded missing perts: 16 of 16
[h5ad] embedded ALL perts for blending: 140 of 140
[h5ad] U_ctrl: (5127, 80)
[genept] reduced: z_avail= 0 u_avail= 4998 dim_z= None dim_u= 64
[exp] U_out = ctrl (replace)
[exp] U_out = concat(+GenePT) -> (5127, 144)
EXP_NAME: baseline_fixed_missing
U_out: (5127, 144) Z_train: (80, 80)


In [449]:
def gate_smoothstep(x, a = GATE_A, b = GATE_B):
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def per_row_weighted_l1_like(delta_true: torch.Tensor, delta_pred: torch.Tensor, eps: float = EPS) -> torch.Tensor:
    """
    Same logic as weighted_l1_like, but returns (N,) per-row.
    Assumes your existing gate_smoothstep(...) exists.
    """
    w = gate_smoothstep(torch.abs(delta_true), a=GATE_A, b=GATE_B)  # (N, G)
    err = torch.abs(delta_pred - delta_true)                        # (N, G)
    num = torch.sum(w * err, dim=1)                                 # (N,)
    den = torch.clamp(torch.sum(w, dim=1), min=eps)                 # (N,)
    return num / den                                                # (N,)

import torch

def weighted_l1_like_rowweighted(
    delta_true: torch.Tensor,     # (N, G)
    delta_pred: torch.Tensor,     # (N, G)
    baseline_wmae: torch.Tensor,  # (N,)
    *,
    eps: float = 1e-8,
    mode: str = "inv_sqrt",       # "inv", "inv_sqrt", "inv_log"
    clamp_min: float = 0.5,
    clamp_max: float = 3.0,
) -> torch.Tensor:
    """
    Row-weighted version of your existing weighted_l1_like.

    Weight idea:
      - baseline_wmae small => ratio metric is unforgiving => upweight that row
      - baseline_wmae large => easier => downweight a bit

    Returns: scalar loss
    """
    # per-row unweighted loss from your current function
    per_row = per_row_weighted_l1_like(delta_true, delta_pred, eps=eps)  # (N,)

    b = baseline_wmae.to(delta_true.device).to(delta_true.dtype)

    if mode == "inv":
        w = 1.0 / (b + eps)
    elif mode == "inv_sqrt":
        w = 1.0 / torch.sqrt(b + eps)
    elif mode == "inv_log":
        w = 1.0 / torch.log1p(b + eps)
    else:
        raise ValueError(f"Unknown mode={mode}")

    # clamp to prevent a few rows from dominating training
    w = torch.clamp(w, min=clamp_min, max=clamp_max)

    # normalized weighted mean (stable)
    return torch.sum(w * per_row) / torch.clamp(torch.sum(w), min=eps)

def weighted_cosine_per_row_torch(dt: torch.Tensor, dp: torch.Tensor, w: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    # dt, dp, w: (B, G)
    wa = w * dt
    wb = w * dp
    num = torch.sum(wa * wb, dim=1)
    da  = torch.sqrt(torch.sum(wa * wa, dim=1))
    db  = torch.sqrt(torch.sum(wb * wb, dim=1))
    denom = torch.clamp(da * db, min=eps)
    return num / denom  # (B,)


In [450]:
class BilinearDeltaModel(nn.Module):
    def __init__(self, d_pert, d_out, rank_r, dropout):
        super().__init__()
        self.rank_r = rank_r

        self.proj_p = nn.Sequential(
            nn.Linear(d_pert, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.proj_o = nn.Sequential(
            nn.Linear(d_out, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # biases
        self.bias_global = nn.Parameter(torch.zeros(1))
        self.bias_gene = None  # set via set_gene_bias

    def set_gene_bias(self, G):
        dev = next(self.parameters()).device
        self.bias_gene = torch.nn.Parameter(torch.zeros(G, device=dev))
        self.bias_global = torch.nn.Parameter(torch.zeros(1, device=dev))

    def forward(self, z_pert, u_out):
        p = self.proj_p(z_pert)    # (B, R)
        o = self.proj_o(u_out)     # (G, R)
        y = p @ o.T                # (B, G)
        y = y + self.bias_gene[None, :] + self.bias_global
        return y

In [451]:
Y = D_train.astype(np.float32)
G = Y.shape[1]
N = Y.shape[0]

Uo_t = torch.tensor(U_out, device=device) # (G, d_out)
Zt = torch.tensor(Z_train, device=device) # (N, d_pert)
Yt = torch.tensor(Y, device=device) # (N, G)

print("N:", N, "G:", G, "device:", device)

N: 80 G: 5127 device: cuda


In [452]:
gt_df = pd.read_csv('Data/training_data_ground_truth_table.csv')
baseline_wmae_t = torch.tensor(
    gt_df["baseline_wmae"].to_numpy(dtype=np.float32),
    device=device,
    dtype=torch.float32,
)

In [453]:
print(f"UOUT_BLEND_ALPHA: {UOUT_BLEND_ALPHA}, ZCTRL_BETA: {ZCTRL_BETA}")
def apply_shrink(pred, baseline, alpha):
    # pred: (B,G) ; baseline: (G,)
    return float(alpha) * pred + (1.0 - float(alpha)) * baseline[None, :]

def train_one_fold(tr_idx, va_idx, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = BilinearDeltaModel(
        d_pert=Zt.shape[1],
        d_out=Uo_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT
    ).to(device)

    model.set_gene_bias(G)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    tr_idx = np.asarray(tr_idx)
    va_idx = np.asarray(va_idx)
    va_idx_t = torch.tensor(va_idx, device=device, dtype=torch.long)

    best_score = -1e18
    best_alpha = 0.0
    best_epoch = 0
    best_state = None
    best_va_pred = None  # unshrunk predictions at best checkpoint
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = tr_idx.copy()
        np.random.shuffle(perm)

        for start in range(0, len(perm), BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t)
            dt_b = Yt.index_select(0, b_t)                 # (B, G)
            bw_b = baseline_wmae_t.index_select(0, b_t)     # (B,)

            loss = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if epoch % 5 == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                va_pred = model(Zt.index_select(0, va_idx_t), Uo_t).detach().cpu().numpy().astype(np.float32)

            va_true = Y[va_idx]

            # alpha sweep (0..0.7) on this fold
            sc_best = -1e18
            a_best = 0.0
            for a in ALPHA_GRID:
                pred_a = apply_shrink(va_pred, delta_baseline, float(a))
                sc = score_delta(va_true, pred_a)["score"]
                if sc > sc_best:
                    sc_best = sc
                    a_best = float(a)

            if sc_best > best_score:
                best_score = sc_best
                best_alpha = a_best
                best_epoch = epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_va_pred = va_pred.copy()
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    return best_score, best_alpha, best_epoch, best_state, best_va_pred

# --- CV: collect OOF preds and optimize a GLOBAL alpha on OOF only ---
kf = KFold(n_splits=8, shuffle=True, random_state=SEED)

oof_pred = np.zeros_like(Y, dtype=np.float32)
oof_hit = np.zeros((N,), dtype=np.int32)

fold_scores = []
fold_alphas = []
fold_epochs = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(N)), 1):
    best_score, best_alpha, best_epoch, best_state, best_va_pred = train_one_fold(tr_idx, va_idx, seed=SEED)
    fold_scores.append(float(best_score))
    fold_alphas.append(float(best_alpha))
    fold_epochs.append(int(best_epoch))

    oof_pred[va_idx] = best_va_pred
    oof_hit[va_idx] += 1

    print(f"fold {fold}: best_score={best_score:.6f} best_alpha={best_alpha:.3f} best_epoch={best_epoch}")

if not np.all(oof_hit == 1):
    print("[warn] OOF coverage not 1 everywhere. min/max:", int(oof_hit.min()), int(oof_hit.max()))

print("cv mean:", float(np.mean(fold_scores)), "std:", float(np.std(fold_scores)))

EPOCHS_MED = int(np.median(fold_epochs))
print("median best_epoch =", EPOCHS_MED)

# Global alpha chosen on OOF predictions only (more stable than per-fold alpha roulette)
best_global_alpha = 0.0
best_global_score = -1e18
for a in ALPHA_GRID:
    pred_a = apply_shrink(oof_pred, delta_baseline, float(a))
    sc = score_delta(Y, pred_a)["score"]
    if sc > best_global_score:
        best_global_score = sc
        best_global_alpha = float(a)

print("OOF global alpha:", best_global_alpha, "OOF score:", best_global_score)

ALPHA_SHRINK = best_global_alpha


UOUT_BLEND_ALPHA: 1, ZCTRL_BETA: 0.5
fold 1: best_score=0.170458 best_alpha=0.860 best_epoch=65
fold 2: best_score=0.117077 best_alpha=0.820 best_epoch=45
fold 3: best_score=0.115384 best_alpha=0.700 best_epoch=20
fold 4: best_score=0.133669 best_alpha=0.860 best_epoch=60
fold 5: best_score=0.164006 best_alpha=0.860 best_epoch=25
fold 6: best_score=0.203510 best_alpha=0.750 best_epoch=65
fold 7: best_score=0.183145 best_alpha=0.750 best_epoch=15
fold 8: best_score=0.128408 best_alpha=0.750 best_epoch=15
cv mean: 0.15195711401499312 std: 0.03074298084479934
median best_epoch = 35
OOF global alpha: 0.778 OOF score: 0.15007080529024108


fold 1: best_score=0.166475 best_alpha=0.860 best_epoch=25
fold 2: best_score=0.115051 best_alpha=0.778 best_epoch=45
fold 3: best_score=0.114235 best_alpha=0.700 best_epoch=25
fold 4: best_score=0.122564 best_alpha=0.860 best_epoch=35
fold 5: best_score=0.159363 best_alpha=0.860 best_epoch=25
fold 6: best_score=0.191482 best_alpha=0.700 best_epoch=30
fold 7: best_score=0.181774 best_alpha=0.750 best_epoch=20
fold 8: best_score=0.128870 best_alpha=0.700 best_epoch=20
cv mean: 0.14747667565431655 std: 0.029022193423455175
median best_epoch = 25
OOF global alpha: 0.778 OOF score: 0.1457519906800826

In [454]:

def fit_full_model(seed, epochs_fixed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = BilinearDeltaModel(
        d_pert=Zt.shape[1],
        d_out=Uo_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT
    ).to(device)

    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    for epoch in range(1, epochs_fixed + 1):
        model.train()
        perm = np.arange(N)
        np.random.shuffle(perm)

        for start in range(0, N, BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t)
            dt_b = Yt.index_select(0, b_t)                 # (B, G)
            bw_b = baseline_wmae_t.index_select(0, b_t)     # (B,)

            loss = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if epoch % EVAL_EVERY == 0 or epoch == epochs_fixed:
            model.eval()
            with torch.no_grad():
                pred_np = model(Zt, Uo_t).detach().cpu().numpy().astype(np.float32)

            pred_np = apply_shrink(pred_np, delta_baseline, ALPHA_SHRINK)
            s = score_delta(Y, pred_np)
            print(f"[seed {seed}] epoch={epoch:4d} train_score={s['score']:.6f} wcos={s['wcos']:.6f} pred_wmae={s['pred_wmae']:.6f} alpha={ALPHA_SHRINK:.3f}")

    model.eval()
    return model

# refit ensemble on ALL 80 perts, using CV-calibrated epoch + OOF-calibrated alpha
models = [fit_full_model(sd, epochs_fixed=EPOCHS_MED) for sd in MODEL_SEEDS]
print("Refit models:", len(models))

def predict_delta_gene(gene_symbol: str) -> np.ndarray:
    z = torch.tensor(emb_pert(gene_symbol)[None, :].astype(np.float32), device=device)
    preds = []
    with torch.no_grad():
        for m in models:
            y = m(z, Uo_t).detach().cpu().numpy().astype(np.float32)[0]
            preds.append(y)
    yhat = np.mean(np.stack(preds, axis=0), axis=0).astype(np.float32)
    yhat = (ALPHA_SHRINK * yhat + (1.0 - ALPHA_SHRINK) * delta_baseline).astype(np.float32)
    return yhat

# Build submission from sample_submission.csv
sub = df_sub.copy()
sub["pert_id"] = sub["pert_id"].astype(str)
sub_gene_cols = [c for c in sub.columns if c != "pert_id"]

# ensure order matches gene_columns
idx = {g: i for i, g in enumerate(gene_columns)}
perm = [idx[g] for g in sub_gene_cols]

# fill default baseline for unknown test perts
sub.loc[:, sub_gene_cols] = np.tile(delta_baseline[perm][None, :], (len(sub), 1))

# fill known val perts (pert_1..pert_60)
hit = 0
for pid, gene in val_map.items():
    vec = predict_delta_gene(gene)[perm]
    m = (sub["pert_id"] == str(pid))
    if m.any():
        sub.loc[m, sub_gene_cols] = vec[None, :]
        hit += int(m.sum())

out_path = "submission_bilinear_refit_oofalpha.csv"
#sub.to_csv(out_path, index=False)
print("[ok] wrote:", out_path, "| filled:", hit)


[seed 6] epoch=  25 train_score=0.197359 wcos=0.492690 pred_wmae=0.071955 alpha=0.778
[seed 6] epoch=  35 train_score=0.203252 wcos=0.496805 pred_wmae=0.071561 alpha=0.778
[seed 7] epoch=  25 train_score=0.198363 wcos=0.494279 pred_wmae=0.071923 alpha=0.778
[seed 7] epoch=  35 train_score=0.204521 wcos=0.498945 pred_wmae=0.071537 alpha=0.778


KeyboardInterrupt: 

[seed 6] epoch=  25 train_score=0.190763 wcos=0.493199 pred_wmae=0.072467 alpha=0.700
[seed 7] epoch=  25 train_score=0.191113 wcos=0.493597 pred_wmae=0.072458 alpha=0.700
[seed 8] epoch=  25 train_score=0.190862 wcos=0.493393 pred_wmae=0.072469 alpha=0.700
Refit models: 3
[ok] wrote: submission_bilinear_refit_oofalpha.csv | filled: 60

epoch=  25 train_score=0.197565 wcos=0.487435 pred_wmae=0.071318
epoch=  50 train_score=0.214566 wcos=0.506001 pred_wmae=0.070564
epoch=  75 train_score=0.255586 wcos=0.546834 pred_wmae=0.068880
epoch= 100 train_score=0.320180 wcos=0.596986 pred_wmae=0.066354
epoch= 125 train_score=0.395301 wcos=0.641472 pred_wmae=0.063575
epoch= 150 train_score=0.465017 wcos=0.673950 pred_wmae=0.061027
epoch= 175 train_score=0.528761 wcos=0.699482 pred_wmae=0.058800
epoch= 200 train_score=0.583847 wcos=0.718520 pred_wmae=0.056899
epoch= 225 train_score=0.634970 wcos=0.734100 pred_wmae=0.055180
epoch= 250 train_score=0.678502 wcos=0.748007 pred_wmae=0.053726
epoch= 275 train_score=0.714378 wcos=0.758516 pred_wmae=0.052467
epoch= 300 train_score=0.750164 wcos=0.767519 pred_wmae=0.051263
epoch= 325 train_score=0.780506 wcos=0.776008 pred_wmae=0.050237
epoch= 350 train_score=0.811266 wcos=0.784896 pred_wmae=0.049284
epoch= 375 train_score=0.837425 wcos=0.791671 pred_wmae=0.048401
epoch= 400 train_score=0.859289 wcos=0.797939 pred_wmae=0.047662
wrote: submission_bilinear.csv